# Alignment Transfer：standalone実行例

T2TT → Attention・CTC・HuBERT → alignment診断 → 5条件のS2UT学習 → 推論・評価を順に確認する旧trainerの実行例です。

> 学習済みbenchmark Baselineとの本比較・ZIPによるDrive保存と復元には[benchmark比較ノートブック](https://colab.research.google.com/github/Yaaamashiro/alignment-transfer-s2ut/blob/main/notebooks/colab/benchmark_comparison.ipynb)を使用します。このノートブックは`--trainer legacy`とDriveへの直接保存を使います。

GPUランタイムで上から実行します。既定の`UPDATES=2`は接続確認用です。実験条件を変更する場合は新しいROOTを使います。

## 1. Drive接続と設定

In [ ]:
#@title 共通ログ：開始・逐次出力・生存通知・完了・失敗
from pathlib import Path
import json
"""Standard-library-only process logging, also embedded in Colab before install."""


def log_event(name, state, logs, detail=""):
    from datetime import datetime, timezone
    from pathlib import Path
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    line = f"{datetime.now(timezone.utc).isoformat()} [{name}] {state} {detail}\n"
    with (root / "notebook-events.log").open("a", encoding="utf-8") as stream:
        stream.write(line)
        stream.flush()
    print(line, end="", flush=True)


def run_process(command, logs, *, name="command", capture_json=False, env=None, cwd=None,
                heartbeat_seconds=10., extra_record=None):
    """Stream every child line; silence triggers liveness, never fake progress.

    stdout JSON is kept apart from stderr. Nested loggers share an owned POSIX
    process group so interruption cannot strand a second training process.
    """
    from datetime import datetime, timezone
    import hashlib
    import json
    import os
    from pathlib import Path
    import queue
    import signal
    import subprocess
    import tempfile
    import threading
    import time

    if heartbeat_seconds <= 0:
        raise ValueError("heartbeat_seconds must be positive")
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    command = list(map(str, command))
    key = hashlib.sha256(json.dumps(name, ensure_ascii=False).encode()).hexdigest()[:12] + "-" + str(time.time_ns())
    output, result_path, receipt = root / (key + ".log"), root / (key + ".stdout.log"), root / (key + ".json")
    launch_env = dict(os.environ if env is None else env)
    launch_env["PYTHONUNBUFFERED"] = "1"
    owns_group = os.name == "posix" and launch_env.get("AT_LOG_MANAGED_GROUP") != "1"
    if owns_group:
        launch_env["AT_LOG_MANAGED_GROUP"] = "1"
    started, clock = datetime.now(timezone.utc).isoformat(), time.perf_counter()
    record = dict(name=name, command=command, started_utc=started,
        host_utc_offset_seconds=int(datetime.now().astimezone().utcoffset().total_seconds()),
        elapsed_seconds=0., returncode=None, status="running", exception=None,
        pid=None, log=str(output.resolve()), stdout_log=str(result_path.resolve()) if capture_json else None,
        gpu_csv=None, gpu_monitor_error=None, cwd=str(Path(cwd or Path.cwd()).resolve()))
    lock, stop, done = threading.Lock(), threading.Event(), threading.Event()
    messages, reader_errors = queue.Queue(maxsize=256), []
    process = reader = result_stream = None
    stream = output.open("w", encoding="utf-8")

    def save_record():
        record["elapsed_seconds"] = time.perf_counter() - clock
        fd, temporary = tempfile.mkstemp(dir=root, prefix=".log-record-")
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as target:
                json.dump(record, target, ensure_ascii=False, indent=2, allow_nan=False)
                target.write("\n")
                target.flush()
            os.replace(temporary, receipt)
        finally:
            if os.path.exists(temporary):
                os.unlink(temporary)

    def notice(state, detail=""):
        line = f"[{name}] {state} pid={record['pid']} elapsed={time.perf_counter()-clock:.1f}s {detail}\n"
        with lock:
            stream.write(line)
            stream.flush()
        print(line, end="", flush=True)

    def read_lines(pipe):
        try:
            for line in pipe:
                with lock:
                    stream.write(line)
                    stream.flush()
                while not stop.is_set():
                    try:
                        messages.put(line, timeout=.1)
                        break
                    except queue.Full:
                        pass
        except Exception as error:
            reader_errors.append(error)
        finally:
            done.set()

    def terminate_child():
        if process is None:
            return
        if owns_group:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
        elif process.poll() is None:
            process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            if owns_group:
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            process.wait()
        if owns_group:
            # A nested child may outlive its exited leader after cancellation.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

    try:
        notice("started", f"full log: {output.resolve()}")
        save_record()
        if capture_json:
            result_stream = result_path.open("w", encoding="utf-8")
        process = subprocess.Popen(command, stdout=result_stream if capture_json else subprocess.PIPE,
            stderr=subprocess.PIPE if capture_json else subprocess.STDOUT, text=True,
            encoding="utf-8", errors="replace", bufsize=1, env=launch_env, cwd=cwd,
            start_new_session=owns_group)
        record["pid"] = process.pid
        notice("running", "waiting for child output")
        save_record()
        pipe = process.stderr if capture_json else process.stdout
        reader = threading.Thread(target=read_lines, args=(pipe,), daemon=True, name="command-log-reader")
        reader.start()
        last_output = last_notice = time.perf_counter()
        while not (done.is_set() and messages.empty() and process.poll() is not None):
            try:
                line = messages.get(timeout=min(.1, heartbeat_seconds))
            except queue.Empty:
                line = None
            if line is not None:
                print(line, end="", flush=True)
                if line.strip():
                    last_output = time.perf_counter()
            if reader_errors:
                raise reader_errors[0]
            now = time.perf_counter()
            if now - max(last_output, last_notice) >= heartbeat_seconds:
                notice("running", f"no_output={now-last_output:.1f}s (liveness only; progress not confirmed)")
                save_record()
                last_notice = now
        code = process.wait()
        if reader_errors:
            raise reader_errors[0]
        if capture_json and code == 0:
            json.loads(result_path.read_text(encoding="utf-8"))
        record.update(returncode=code, status="succeeded" if code == 0 else "failed_or_interrupted")
        if code:
            raise subprocess.CalledProcessError(code, command)
        notice("completed", f"exit={code}; full log: {output.resolve()}")
    except BaseException as error:
        record.update(status="failed_or_interrupted", exception=type(error).__name__ + ": " + str(error))
        terminate_child()
        if process is not None:
            record["returncode"] = process.returncode
        notice("interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
               f"exit={record['returncode']} {type(error).__name__}; full log: {output.resolve()}")
        raise
    finally:
        stop.set()
        if reader is not None:
            reader.join(timeout=2)
        if process is not None and (reader is None or not reader.is_alive()):
            pipe = process.stderr if capture_json else process.stdout
            if pipe is not None:
                pipe.close()
        if result_stream is not None:
            result_stream.close()
        if extra_record is not None:
            record.update(extra_record() if callable(extra_record) else extra_record)
        save_record()
        stream.close()
    result = subprocess.CompletedProcess(command, record["returncode"],
        result_path.read_text(encoding="utf-8") if capture_json else None)
    result.record = record
    return result


def execute(*arguments, name=None, capture_json=False, env=None, cwd=None):
    command = list(map(str, arguments))
    if name is None:
        name = command[2].rsplit('.', 1)[-1] if len(command) > 2 and command[1] == '-m' else Path(command[0]).name
        if '--method' in command:
            name += ':' + command[command.index('--method') + 1]
        if '--prior-type' in command:
            name += ':' + command[command.index('--prior-type') + 1]
    return run_process(command, LOGS, name=name, capture_json=capture_json, env=env, cwd=cwd)


def execute_json(*arguments, name=None, env=None, cwd=None):
    return json.loads(execute(*arguments, name=name, capture_json=True, env=env, cwd=cwd).stdout)

LOGS = Path('/content/alignment-transfer-logs/pipeline/bootstrap')


In [ ]:
from google.colab import drive
log_event('drive-mount', 'started', LOGS)
try:
    drive.mount('/content/drive')
except BaseException as error:
    log_event('drive-mount', 'failed', LOGS, type(error).__name__)
    raise
else:
    log_event('drive-mount', 'completed', LOGS)
import os
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git' #@param {type:"string"}
REPO_REVISION = '107605f4f5bd7b53722d73ced3fa5a76560e7a9d' #@param {type:"string"}
ROOT = '/content/drive/MyDrive/alignment-transfer-s2ut-data/experiment-stream-001' #@param {type:"string"}
CORPUS = '/content/drive/MyDrive/ja-en-direct-s2st-corpus-data' #@param {type:"string"}
RECIPE = 'controlled' #@param ['controlled', 'strong']
RUNS = ROOT + '/' + RECIPE
ANNEAL_STEPS = 10000 #@param {type:"integer"}
UPDATES = 2 #@param {type:"integer"}
PYTHON = '/content/alignment-venv/bin/python'
LOGS = Path('/content/alignment-transfer-logs/pipeline') / Path(ROOT).name / RECIPE
os.environ['HF_HOME'] = '/content/huggingface'
os.environ['MPLBACKEND'] = 'Agg'
os.environ['PYTHONUNBUFFERED'] = '1'


## 2. 環境の構築と検査
すべてのコマンドは開始・PID・途中出力・完了／失敗を表示し、`LOGS`へ保存します。無出力が10秒続いた場合は生存通知を表示します。生存通知だけで実際の進捗が確認できたとは扱いません。


In [ ]:
# Clone/install only. Never point outputs at the corpus or benchmark repositories.
import subprocess, sys

from pathlib import Path
assert len(REPO_REVISION) == 40 and 'SET_' not in REPO_URL
assert not Path(ROOT).resolve().is_relative_to(Path(CORPUS).resolve())
if not Path('/content/alignment-transfer-s2ut/.git').exists():
    execute('git', 'clone', '--no-checkout', REPO_URL, '/content/alignment-transfer-s2ut')
execute('git', '-C', '/content/alignment-transfer-s2ut', 'fetch', 'origin', REPO_REVISION)
execute('git', '-C', '/content/alignment-transfer-s2ut', 'checkout', '--detach', REPO_REVISION)
os.chdir('/content/alignment-transfer-s2ut')
execute(
    sys.executable, '-m', 'pip', 'install', 'uv'
)
execute(
    'uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv'
)
execute(
    PYTHON, '-m', 'pip', 'install', 'pip==23.3.2', 'setuptools<81', 'wheel'
)
execute(
    PYTHON, '-m', 'pip', 'install', 'torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4',
    'Cython==3.2.9'
)
execute(
    PYTHON, '-m', 'pip', 'install', '--no-build-isolation', '-r', 'requirements-fairseq.txt'
)
execute(
    PYTHON, '-m', 'pip', 'install', '-r', 'requirements.txt'
)
execute(
    PYTHON, '-m', 'pytest', '-v', '--tb=short'
)
execute(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'runtime')
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')


## 3. Manifestと共通構成の確認

In [ ]:
# First import only. On resume, retain the existing canonical manifest unchanged.
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.prepare_manifest', '--input',
    f'{CORPUS}/production/manifests/releases/accepted.jsonl', '--corpus-root', f'{CORPUS}',
    '--output', f'{ROOT}/manifest.jsonl', '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.compare_architectures', '--t2tt-config',
    'configs/t2tt/fisher.yaml', '--s2ut-config', 'configs/s2ut/fisher.yaml', '--output',
    f'{ROOT}/architecture.json', '--resume'
)


## 4. T2TT学習

In [ ]:
# Keep the same recipe; resume automatically if a checkpoint exists.
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.train_t2tt', '--manifest', f'{ROOT}/manifest.jsonl',
    '--config', 'configs/t2tt/fisher.yaml', '--output', f'{ROOT}/t2tt', '--max-updates',
    f'{UPDATES}', '--device', 'cuda', '--resume-if-exists'
)


## 5. Attention・CTC・unitとteacherの作成

In [ ]:
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.extract_t2tt_attention', '--manifest',
    f'{ROOT}/manifest.jsonl', '--checkpoint', f'{ROOT}/t2tt/checkpoint_best.pt', '--output',
    f'{ROOT}/mt', '--device', 'cuda', '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.align_speech_text', '--manifest',
    f'{ROOT}/manifest.jsonl', '--config', 'configs/alignment/ctc.yaml', '--output', f'{ROOT}/ctc',
    '--device', 'cuda', '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.fetch_assets', '--config',
    'configs/units/hubert_km100.yaml'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.extract_units', '--manifest', f'{ROOT}/manifest.jsonl',
    '--config', 'configs/units/hubert_km100.yaml', '--output', f'{ROOT}/units', '--device', 'cuda',
    '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.build_teacher_alignment', '--manifest',
    f'{ROOT}/manifest.jsonl', '--mt-attention', f'{ROOT}/mt', '--alignment', f'{ROOT}/ctc',
    '--units', f'{ROOT}/units', '--config', 'configs/s2ut/fisher.yaml', '--output',
    f'{ROOT}/teacher', '--resume'
)


## 6. Alignment診断

In [ ]:
# Audit coverage/uncertainty before training; inspect heatmaps against original audio.
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.diagnose_alignment', '--manifest',
    f'{ROOT}/manifest.jsonl', '--teacher', f'{ROOT}/teacher', '--units', f'{ROOT}/units',
    '--config', 'configs/s2ut/fisher.yaml', '--split', 'dev', '--max-samples', '500', '--heatmaps',
    '20', '--source-mapping', 'frontend', '--prior-smoothing', '0.05', '--output',
    f'{ROOT}/alignment-audit', '--overwrite'
)


`RECIPE`でcontrolled / strongを選択します。strongは文字補助decoder・decoder CTC・CMVN・SpecAugmentを使用し、両recipeとも単調prior対照を含めて比較します。出力先はrecipe別です。UPDATES=2は接続確認、pilotは5000〜20000更新、本学習は全条件共通の更新数を設定してください。過去の実装からのresumeはcontractの不一致で停止するため、新しいROOTを使用します。

## 7. 5条件のS2UT学習

In [ ]:
# One recipe, timing and smoothing across all five conditions.
SHARED = ['--trainer', 'legacy', '--manifest', f'{ROOT}/manifest.jsonl',
    '--config', 'configs/s2ut/fisher.yaml', '--units', f'{ROOT}/units',
    '--loader-workers', 2, '--prefetch-batches', 2, '--cache-dir', '/content/alignment-transfer-cache',
    '--cache-gb', 8, '--pin-memory', '--recipe', RECIPE, '--prior-smoothing', 0.05,
    '--source-mapping', 'frontend', '--anneal-steps', ANNEAL_STEPS,
    '--validation-interval', 1000, '--save-interval-updates', 1000,
    '--max-updates', UPDATES, '--device', 'cuda', '--resume-if-exists']
CONDITIONS = [('baseline', 'baseline', 't2tt'), ('distillation', 'distillation', 't2tt'),
    ('guidance', 'guidance', 't2tt'), ('annealed_guidance', 'annealed_guidance', 't2tt'),
    ('annealed_monotonic', 'annealed_guidance', 'monotonic')]
for name, method, prior in CONDITIONS:
    teacher = ['--teacher', f'{ROOT}/teacher'] if name not in ('baseline', 'annealed_monotonic') else []
    control = ['--prior-type', 'monotonic'] if prior == 'monotonic' else []
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.train_s2ut', *SHARED,
        '--method', method, '--output', f'{RUNS}/{name}', *teacher, *control)
execute(PYTHON, '-m', 'alignment_transfer_s2ut.compare_experiments',
    *[f'{RUNS}/{name}' for name, _, _ in CONDITIONS])


推論入力には`pair_id, ja_audio`だけを持つJSONLを用意します。英語referenceやtranscriptを含む学習manifestは渡しません。評価に使うsplitとID集合を一致させてください。

## 8. 推論と最終評価

**最終評価用です。** 学習条件の調整にはdevを使い、testを選択基準にしません。2更新の接続確認結果は研究上のスコアとして扱いません。

In [ ]:
SPEECH_ONLY = ROOT + '/test_speech_only.jsonl' #@param {type:"string"}
METHOD = 'annealed_guidance' #@param ['baseline', 'distillation', 'guidance', 'annealed_guidance', 'annealed_monotonic']
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.prepare_inference_input', '--manifest',
    f'{ROOT}/manifest.jsonl', '--split', 'test', '--output', f'{SPEECH_ONLY}', '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/s2ut/vocoder.yaml'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.infer', '--input', f'{SPEECH_ONLY}', '--checkpoint',
    f'{RUNS}/{METHOD}/checkpoint_best.pt', '--vocoder-config', 'configs/s2ut/vocoder.yaml',
    '--output', f'{RUNS}/{METHOD}/test-predictions.jsonl', '--device', 'cuda', '--resume'
)
execute(
    PYTHON, '-m', 'alignment_transfer_s2ut.evaluate', '--manifest', f'{ROOT}/manifest.jsonl',
    '--split', 'test', '--predictions', f'{RUNS}/{METHOD}/test-predictions.jsonl', '--units',
    f'{ROOT}/units', '--asr-config', 'configs/experiments/evaluation.yaml', '--output',
    f'{RUNS}/{METHOD}/test-metrics.json', '--device', 'cuda', '--resume'
)


## 保存結果と再開

`ROOT/t2tt/`にteacher、`ROOT/teacher/`に投影済みalignment、`ROOT/<recipe>/<condition>/`に各学習結果を保存します。各CLIは失敗時に停止します。同じ設定・ROOTで再実行すると、完成済みartifactとcheckpointを再利用します。

この旧経路はcheckpointをDriveへ直接保存します。本比較ではbenchmark比較ノートブックのローカル作業・ZIP保存を使用してください。